# Trabalho Final — Avaliação de Políticas Públicas com Dados (APPD)

**Programa:** Mestrado Profissional em Administração Pública — IDP
**Professor:** Hélio Bomfim de Macêdo Filho
**Integrantes:** [PREENCHER: nomes completos]
**Política avaliada:** Programa Academia da Saúde (Portaria GM/MS nº 719/2011; Portaria GM/MS nº 2.681/2013)

**Pergunta causal:** a implantação do primeiro polo do Programa Academia da Saúde reduziu as internações por doenças cardiovasculares e metabólicas sensíveis à atividade física (doenças hipertensivas, AVC e diabetes) nos municípios que o adotaram?

**Estimando:** ATT — efeito médio sobre os municípios que implantaram ao menos um polo.
**Unidade de análise:** município × ano, 2008–2024.
**Métodos:** escore de propensão (Módulo 03) e diferenças-em-diferenças com estudo de eventos, TWFE e Callaway & Sant'Anna (Módulo 04).

> **Declaração de uso de IA (versão preliminar, a atualizar na Fase 8).** Este caderno foi desenvolvido com apoio do Claude Code (Anthropic), modelo [PREENCHER: nome e versão], para gerar e depurar código. Os autores revisaram cada célula, reexecutaram integralmente o caderno e assumem responsabilidade pelo conteúdo. Prompts e saídas relevantes estão em `docs/ia/`.

**Caderno 01 — extração:** download, descompressão e seleção, dicionários, agregação e gravação em `data/processed/`.

## 1. Identificação

Este caderno cobre as etapas 3 a 6 das orientações (download, descompressão e seleção, dicionários, construção e validação). A análise fica no caderno `02_analise.ipynb`.

**Fontes** (detalhes no PLANO.md, seção 5): CNES-ST (tratamento), CNES-IN (tratamento alternativo), CNES-EP e CNES-LT (covariáveis de 2010), SIH/SUS-RD (desfechos e placebo), SIM (opcional), IBGE (municípios, população, PIB, Censo 2010).

**Tempo estimado de execução:** [PREENCHER após a primeira execução completa].

## 2. Preparação do ambiente

**O que:** instala as dependências com as versões fixadas em `requirements.txt`, importa as bibliotecas, define caminhos relativos à raiz do repositório e a semente aleatória.
**Por quê:** reprodutibilidade — qualquer pessoa deve conseguir rodar o caderno com "Restart & Run All" a partir de `notebooks/`.

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import time
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

RAIZ = Path("..")
sys.path.insert(0, str(RAIZ / "src"))

DIR_RAW = RAIZ / "data" / "raw"
DIR_PROC = RAIZ / "data" / "processed"
DIR_DIC = DIR_PROC / "dicionarios"
DIR_FIG = RAIZ / "outputs" / "figuras"
DIR_TAB = RAIZ / "outputs" / "tabelas"
for d in (DIR_RAW, DIR_PROC, DIR_DIC, DIR_FIG, DIR_TAB):
    d.mkdir(parents=True, exist_ok=True)

SEMENTE = 42
np.random.seed(SEMENTE)
pd.set_option("display.max_columns", 30)
print("Python", sys.version.split()[0], "| pandas", pd.__version__)

from appd import download as dl
from appd import agregacao as ag

dl.configurar_cache(RAIZ)

UFS = ["AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", "MA", "MG", "MS", "MT", "PA",
       "PB", "PE", "PI", "PR", "RJ", "RN", "RO", "RR", "RS", "SC", "SE", "SP", "TO"]
ANOS = list(range(2008, 2025))
MES_CNES = 12  # uma competência por ano (dezembro), PLANO.md seção 5
INICIO = time.time()

As constantes `UFS`, `ANOS` e `MES_CNES` definem o recorte. O cache do PySUS fica em `data/raw/pysus` e os brutos são apagados após a agregação.

## 3. Download dos dados brutos

Todas as chamadas passam por `src/appd/download.py`, que usa o FTP do DATASUS (`source="origin"`) e, em caso de falha, o catálogo-espelho do PySUS, registrando a origem efetiva e a data de acesso em `dl.registro_acessos()`.

### 3.1 Lista de municípios (API de localidades do IBGE)
**Por quê:** chave mestra do painel (código de 7 dígitos, `cod6` do DATASUS, UF e região).

In [ ]:
municipios = dl.municipios_ibge()
assert municipios["cod_ibge7"].is_unique
assert municipios["cod6"].is_unique
print(len(municipios), "municípios")
municipios.head()

**Resultado:** [PREENCHER: número de municípios e conferência com o número oficial vigente]

### 3.2 CNES-ST: polos tipo 74 por município × ano
**Por quê:** constrói o tratamento D. Para cada UF e ano lê só `CNES`, `CODUFMUN` e `TP_UNID` (texto) da competência de dezembro e conta os estabelecimentos tipo 74.

In [ ]:
COLS_ST = ["CNES", "CODUFMUN", "TP_UNID"]
partes_polos = []
for ano in ANOS:
    for uf in UFS:
        st = dl.baixar_datasus("cnes", uf, ano, MES_CNES, grupo="ST", colunas=COLS_ST)
        partes_polos.append(ag.contar_polos(st, ano))
polos = pd.concat(partes_polos, ignore_index=True)
assert not polos.duplicated(["cod6", "ano"]).any()
polos.groupby("ano")["n_polos"].agg(["count", "sum"])

**Resultado:** [PREENCHER: primeiro ano com polos tipo 74; evolução do número de municípios com polo]

### 3.3 CNES-IN (incentivos 81.12/81.13), CNES-EP e CNES-LT (dez/2010)
*A implementar na Fase 3, após verificar o layout dos grupos IN, EP e LT no teste de fumaça.*

### 3.4 SIH/SUS-RD: internações por município de residência × ano × grupo de CID
**Por quê:** constrói os desfechos (hipertensão, AVC, diabetes) e o placebo (apendicite). Processa UF por UF e mês por mês, lê só as colunas necessárias, agrega na hora e apaga o bruto.

> Etapa mais demorada do trabalho (≈ 5.500 arquivos). Antes de rodar tudo, valide com uma UF pequena (AC ou RR) contra o TabNet.

In [ ]:
COLS_RD = ["MUNIC_RES", "DIAG_PRINC", "ANO_CMPT", "IDADE", "COD_IDADE"]
partes_sih = []
for ano in ANOS:
    for uf in UFS:
        for mes in range(1, 13):
            rd = dl.baixar_datasus("sih", uf, ano, mes, grupo="RD", colunas=COLS_RD)
            partes_sih.append(ag.agregar_sih(rd))
sih = (pd.concat(partes_sih, ignore_index=True)
         .groupby(["cod6", "ano", "grupo"], as_index=False)["internacoes"].sum())
assert not sih.duplicated(["cod6", "ano", "grupo"]).any()
sih.pivot_table(index="ano", columns="grupo", values="internacoes", aggfunc="sum")

**Resultado:** [PREENCHER: totais nacionais por ano e grupo; conferência de um município-ano contra o TabNet]

**Atenção:** `ANO_CMPT` é o ano de competência (processamento) da AIH; o arquivo de janeiro de 2009 pode conter internações de 2008. A decisão sobre usar `ANO_CMPT` ou a data de internação deve ser registrada aqui.

### 3.5 IBGE via SIDRA: população, PIB municipal 2010 e Censo 2010
*A implementar na Fase 3, após confirmar os números das tabelas no teste de fumaça (estimativas populacionais, PIB dos municípios, Censo 2010).*

## 4. Descompressão e seleção

O PySUS converte os `.dbc` do DATASUS em parquet no cache local. Mantemos apenas as colunas listadas em `COLS_ST` e `COLS_RD` (e equivalentes das demais bases) e apagamos os brutos após a agregação. *Lista explícita dos arquivos mantidos: a preencher na Fase 3.*

## 5. Dicionário de variáveis e rótulos de valores

Exportados **antes** da transformação para `data/processed/dicionarios/<base>_colunas.csv` (nome, tipo, descrição) e rótulos relevantes (`TP_UNID`, grupos de CID). *A implementar na Fase 3.*

In [ ]:
rotulos_cid = pd.DataFrame(
    [(g, ini, fim) for g, (ini, fim) in ag.GRUPOS_CID.items()],
    columns=["grupo", "cid_inicial", "cid_final"],
)
rotulos_cid.to_csv(DIR_DIC / "rotulos_grupos_cid.csv", index=False)
rotulos_cid

Os grupos de CID-10 usados como desfecho e placebo ficam registrados em `rotulos_grupos_cid.csv`.

## 6. Agregação e validação

Junta polos, internações e a lista de municípios em um painel município × ano e valida com `assert`: chave única, número esperado de linhas e ausência de faltantes inesperados. *A implementar na Fase 3.*

## 7. Gravação em `data/processed/`

Grava `painel_bruto.parquet`, `covariaveis_2010.csv` e o registro de acessos (datas e origens efetivas). *A implementar na Fase 3.*

In [ ]:
dl.registro_acessos().to_csv(DIR_PROC / "registro_acessos.csv", index=False)
print(f"Tempo total: {(time.time() - INICIO) / 60:.1f} min")

O tempo total medido deve ser copiado para o `README.md`.